# BONUS: Timestamps, UTC & Auto Loader with File Archiving

Bonus notebook for Data Engineers working in Databricks.

Covers two key topics:
1. **Working with dates and timestamps** in Spark SQL / PySpark — timezone handling, UTC standardization, and `session.timeZone` pitfalls
2. **Auto Loader with file archiving** — using `cloudFiles.cleanSource = MOVE` to automatically archive processed files, plus a `foreachBatch` alternative

**Prerequisites:** Day 2 — Streaming & Incremental Ingestion

## Part 1: Timestamps and UTC

---

### Setup

Environment configuration — imports, environment variables, paths.

In [0]:
%run ../setup/00_setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, DateType, LongType
)
from datetime import datetime, timezone

# ── KLUCZOWE USTAWIENIE: Spark zawsze operuje na UTC wewnętrznie.
# Ustawienie session.timeZone na UTC eliminuje niejednoznaczności
# przy wyświetlaniu i eksporcie danych między strefami czasowymi.
spark.conf.set("spark.sql.session.timeZone", "UTC")

# ── Ścieżki do demo archiwizacji (Part 2)
BONUS_BASE       = f"{DATASET_PATH}/bonus_demo"
SOURCE_PATH      = f"{BONUS_BASE}/source"
ARCHIVE_PATH     = f"{BONUS_BASE}/archive"
CHECKPOINT_PATH  = f"{BONUS_BASE}/checkpoint_archive"
SCHEMA_PATH      = f"{BONUS_BASE}/schema_archive"
TARGET_TABLE     = f"{CATALOG}.{BRONZE_SCHEMA}.bonus_orders_archive"

# Cleanup poprzedniego uruchomienia
dbutils.fs.rm(BONUS_BASE, True)
print(f"  Środowisko przygotowane: {BONUS_BASE}")
print(f"    spark.sql.session.timeZone = {spark.conf.get('spark.sql.session.timeZone')}")

### 1.1 Date vs Timestamp — Key Differences

| Type | Precision | Timezone | Use Case |
|------|-----------|----------|----------|
| `DATE` | day | none | calendar dates, partitioning |
| `TIMESTAMP` | microseconds | session timezone dependent | event logging, CDC, SLA |
| `TIMESTAMP_NTZ` | microseconds | **none — always naive** | UTC storage, avoid ambiguity |

**Best practice:** Store all timestamps as UTC. Convert to local only for display.

In [0]:
# Input data — events from different systems and time zones
raw_data = [
    # (id, event,          timestamp as string,         source timezone)
    (1, "order PL", "2024-06-15 14:30:00",        "Europe/Warsaw"),
    (2, "zamówienie US", "2024-06-15 08:30:00",        "America/New_York"),
    (3, "zamówienie JP", "2024-06-15 21:30:00",        "Asia/Tokyo"),
    (4, "zamówienie UK", "2024-06-15 13:30:00",        "Europe/London"),
    (5, "zamówienie AU", "2024-06-15 23:30:00",        "Australia/Sydney"),
]

schema = StructType([
    StructField("id",            LongType(),   False),
    StructField("event",         StringType(), True),
    StructField("local_time_str", StringType(), True),
    StructField("source_tz",     StringType(), True),
])

df_raw = spark.createDataFrame(raw_data, schema)

# Parsujemy string do TIMESTAMP (Spark zakłada session.timeZone = UTC)
df_raw = df_raw.withColumn(
    "local_timestamp",
    F.to_timestamp("local_time_str", "yyyy-MM-dd HH:mm:ss")
)

display(df_raw)

### 1.2 Converting Timezones to UTC

**Problem:** Each source system delivered a timestamp in its local timezone. We need to normalize all of them to UTC.

`from_utc_timestamp(ts, tz)` — converts UTC to a local timezone (for display)
`to_utc_timestamp(ts, tz)` — converts a local timezone to UTC (for storage)

In [0]:
# ── to_utc_timestamp(col, tz): treats the column as time in the given timezone
# and converts it to UTC. Use this pattern for inbound data.
df_utc = (
    df_raw
    # For each record we know the source timezonefę źródłową — przeliczamy do UTC
    .withColumn(
        "event_utc",
        F.to_utc_timestamp("local_timestamp", F.col("source_tz"))
    )
    # Kolumna daty (tylko dzień) — przydatna do partycjonowania
    .withColumn("event_date", F.to_date("event_utc"))
)

display(
    df_utc.select(
        "id", "event", "source_tz",
        "local_time_str",
        "event_utc",
        "event_date"
    )
)

### 1.3 Displaying Local Time — `from_utc_timestamp`

Data is stored in UTC. When a user in a local office wants to see local time, convert **only at display time** — do not change the stored UTC value.

```python
# Pattern: store UTC, convert for presentation only
df.withColumn('local_time', F.from_utc_timestamp('event_utc', 'Europe/Warsaw'))
```

In [0]:
# ── from_utc_timestamp(col, tz): converts UTC → target local timezone
# Pattern: store UTC, convert only for presentation

TARGET_TIMEZONES = {
    "PL (Warsaw)":   "Europe/Warsaw",
    "US (New York)": "America/New_York",
    "JP (Tokyo)":    "Asia/Tokyo",
}

df_display = df_utc.select("id", "event", "event_utc")

for label, tz in TARGET_TIMEZONES.items():
    col_name = f"display_{label.split(' ')[0].lower()}"
    df_display = df_display.withColumn(
        col_name,
        F.date_format(
            F.from_utc_timestamp("event_utc", tz),
            "yyyy-MM-dd HH:mm:ss z"
        )
    )

display(df_display)

### 1.4 Pitfall: `session.timeZone` and String Parsing

If `spark.sql.session.timeZone` is not set to `UTC`, Spark interprets timezone-naive timestamp strings as the **cluster's local time** — not UTC.

**Fix:** Always set `spark.conf.set('spark.sql.session.timeZone', 'UTC')` at cluster config level.

> **Exam tip:** `TIMESTAMP_NTZ` is unaffected by session timezone.

In [0]:
TS_STRING = "2024-06-15 12:00:00"  # brak informacji o strefie

# ── Demonstrate the effect of session.timeZone on string parsing
results = []
for tz_setting in ["UTC", "America/New_York", "Asia/Tokyo", "Europe/Warsaw"]:
    spark.conf.set("spark.sql.session.timeZone", tz_setting)
    val = spark.sql(f"SELECT CAST('{TS_STRING}' AS TIMESTAMP) AS ts").collect()[0]["ts"]
    results.append((tz_setting, str(val)))

# Przywracamy UTC (standard)
spark.conf.set("spark.sql.session.timeZone", "UTC")

df_tz_demo = spark.createDataFrame(results, ["session_timeZone", "parsed_timestamp_utc_repr"])
display(df_tz_demo)

print("\n  Przywrócono spark.sql.session.timeZone = UTC")

### 1.5 Best Practices — Delta Table with UTC

Recommended schema for production tables accepting events from multiple timezones:

```
event_id          BIGINT
event_timestamp   TIMESTAMP     -- stored as UTC
local_timestamp   TIMESTAMP     -- optional, for display only
date_partition    DATE          -- derived from UTC timestamp
ingested_at       TIMESTAMP     -- watermark for CDC
```

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

# Create Delta table with the recommended UTC schema
spark.sql("DROP TABLE IF EXISTS bonus_events_utc")
spark.sql("""
    CREATE TABLE IF NOT EXISTS bonus_events_utc (
        event_id        BIGINT,
        event_timestamp TIMESTAMP,   -- zawsze UTC
        event_date      DATE,        -- partycja po UTC date
        source_timezone STRING,      -- strefa źródłowa jako metadane
        event_name      STRING,
        ingested_at     TIMESTAMP    -- czas ingestii (UTC)
    )
    USING DELTA
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'false',
        'comment' = 'Zdarzenia z wielu stref — timestamp zawsze UTC'
    )
""")

# Wstawiamy dane z wcześniej przygotowanego DataFrame
df_final = (
    df_utc
    .withColumnRenamed("id", "event_id")
    .withColumnRenamed("event", "event_name")
    .withColumnRenamed("event_utc", "event_timestamp")
    .withColumn("ingested_at", F.current_timestamp())
    .select("event_id", "event_timestamp", "event_date",
            "source_tz", "event_name", "ingested_at")
    .withColumnRenamed("source_tz", "source_timezone")
)

df_final.write.mode("append").saveAsTable("bonus_events_utc")

display(spark.sql("""
    SELECT
        event_id,
        event_name,
        source_timezone,
        event_timestamp                                          AS utc_time,
        from_utc_timestamp(event_timestamp, 'Europe/Warsaw')    AS warsaw_time,
        from_utc_timestamp(event_timestamp, 'America/New_York') AS new_york_time,
        from_utc_timestamp(event_timestamp, 'Asia/Tokyo')       AS tokyo_time
    FROM bonus_events_utc
    ORDER BY event_id
"""))

---

## Part 2: Auto Loader with File Archiving

---

### Scenario

An order system drops JSON files into a `source/` folder every few minutes. After processing, files must be **moved to `archive/`** to prevent reprocessing.

We will use Auto Loader's built-in `cloudFiles.cleanSource = MOVE` option.

### 2.1 Prepare Simulation Data

Create 3 micro-batch JSON files simulating incoming orders.

In [0]:
import json

# Load existing order files from the streaming dataset and split into 3 batches
SOURCE_STREAM_FILES = f"{DATASET_PATH}/orders/stream/*.json"
df_orders = spark.read.json(SOURCE_STREAM_FILES)
batches = df_orders.randomSplit([0.33, 0.33, 0.34], seed=42)

for i, batch_df in enumerate(batches, start=1):
    batch_df.coalesce(1).write.mode("overwrite").json(f"{SOURCE_PATH}/batch_{i:02d}")
    count = batch_df.count()
    print(f"  batch_{i:02d}: {count} rekordów → {SOURCE_PATH}/batch_{i:02d}")

# Sprawdzamy co jest w source
print("\nZawartość source/:")
for f in dbutils.fs.ls(SOURCE_PATH):
    print(f"  {f.path}")

### 2.2 Auto Loader with `cloudFiles.cleanSource = MOVE`

Key options:

- `cloudFiles.cleanSource` = `"MOVE"` — automatically moves files after processing
- `cloudFiles.cleanSource.moveDestination` = destination path for archived files
- Works only with Databricks Auto Loader (`readStream.format("cloudFiles")`)
- Requires `trigger(availableNow=True)` or `trigger(processingTime=...)`

> **Exam tip:** `cleanSource=DELETE` permanently deletes processed files; `cleanSource=MOVE` archives them.

In [0]:
from pyspark.sql.streaming import StreamingQuery

# ── Create Auto Loader stream with file archiving
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",                    "json")
    # ── Schemat i typy
    .option("cloudFiles.schemaLocation",            SCHEMA_PATH)
    .option("cloudFiles.inferColumnTypes",          "true")
    # ── Archiwizacja: przenieś przetworzone pliki do ARCHIVE_PATH
    .option("cloudFiles.cleanSource",               "MOVE")
    .option("cloudFiles.cleanSource.moveDestination", ARCHIVE_PATH)
    .option("cloudFiles.cleanSource.retentionDuration", "0 hours")  # DBR 16.4+
    # ── Kontrola przepustowości
    .option("cloudFiles.maxFilesPerTrigger",        "1")
    # ── Dodaje kolumnę _metadata z nazwą pliku, rozmiarem, czasem modyfikacji
    .load(SOURCE_PATH)
    # Dodajemy timestamp ingestii (UTC)
    .withColumn("ingested_at", F.current_timestamp())
    # Wyciągamy nazwę pliku źródłowego z metadanych
    .withColumn("source_file", F.col("_metadata.file_path"))
)

# ── Uruchamiamy w trybie Triggered (AvailableNow) — przetwarza wszystkie
# dostępne pliki i kończy się. Idealne do batch-like jobs w potoku.
query: StreamingQuery = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .table(TARGET_TABLE)
)

query.awaitTermination()
print(f" Stream zakończony. Przetworzone rekordy zapisane do: {TARGET_TABLE}")

### 2.3 Verification — Table Data and Archived Files

In [0]:
# ── 1. How many records in the target table?
count = spark.table(TARGET_TABLE).count()
print(f"Rekordów w tabeli '{TARGET_TABLE}': {count}")

# ── 2. Preview data with source_file and ingested_at columns
display(
    spark.table(TARGET_TABLE)
    .limit(10)
)

# ── 3. Sprawdzamy czy source/ jest pusty (pliki przeniesione)
print("\nZawartość source/ po przetworzeniu:")
try:
    remaining = dbutils.fs.ls(SOURCE_PATH)
    for f in remaining:
        print(f"  {f.path}")
except Exception:
    print("  (folder pusty lub brak plików JSON)")

# ── 4. Sprawdzamy archiwum
print("\nZawartość archive/:")
for f in dbutils.fs.ls(ARCHIVE_PATH):
    print(f"  {f.path}")

### 2.4 Stream State — `cloud_files_state`

Databricks exposes a table-valued function `cloud_files_state(checkpoint_path)` that returns the state of each file: `queued`, `processed`, or `archived`.

In [0]:
# cloud_files_state — TVF returns metadata about each file processed by Auto Loader
# Columns: path, timestamp, batchId, processedTimestamp, commit_time, size
display(
    spark.sql(f"""
        SELECT
            path,
            size,
            batchId,
            timestamp       AS file_modification_time,
            commit_time
        FROM cloud_files_state('{CHECKPOINT_PATH}')
        ORDER BY batchId, path
    """)
)

### 2.5 Alternative: Manual Archiving via `foreachBatch`

When you need **custom archiving logic** (e.g., `archive/YYYY/MM/DD/` folders, different rules per file type, audit logging), implement archiving inside `foreachBatch`.

```python
def archive_batch(batch_df, batch_id):
    batch_df.write.format('delta').mode('append').saveAsTable(TARGET_TABLE)
    # custom archive logic here
    dbutils.fs.mv(source_path, archive_path)

stream = df_stream.writeStream.foreachBatch(archive_batch) \
    .option('checkpointLocation', checkpoint) \
    .trigger(availableNow=True).start()
```

---

## Cleanup

In [0]:
# Clean up resources after demo
spark.sql("DROP TABLE IF EXISTS bonus_events_utc")
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE_V2}")
dbutils.fs.rm(BONUS_BASE, True)
print("✅  Cleanup zakończony")

---

## Summary — Cheatsheet

### Timestamps and UTC

| Task | Spark / SQL Function |
|------|---------------------|
| Set UTC as cluster standard | `spark.conf.set("spark.sql.session.timeZone", "UTC")` |
| Parse timezone-naive string → UTC | `to_timestamp(col) + session.timeZone=UTC` |
| Convert local → UTC | `to_utc_timestamp(col, 'Europe/Warsaw')` |
| Convert UTC → local | `from_utc_timestamp(col, 'Europe/Warsaw')` |
| Timezone-naive type | `TIMESTAMP_NTZ` |

### Auto Loader File Archiving

| Option | Behavior |
|--------|----------|
| `cleanSource=MOVE` | Move processed files to `moveDestination` |
| `cleanSource=DELETE` | Delete processed files permanently |
| `cloud_files_state()` | TVF: show state of each file (queued/processed/archived) |
| `foreachBatch` | Custom logic per micro-batch |

> **Exam relevance:** Auto Loader is the preferred pattern for incremental cloud file ingestion. File archiving prevents duplicate processing on restart.